In [1]:
%load_ext aiida
%aiida

Loaded AiiDA DB environment - profile name: bit.

In [2]:
from aiida_vasp.workchains.v2 import VaspRelaxUpdater, VaspBuilderUpdater
from aiida_grouppathx import GroupPathX, decorate_with_exit_status
from ase.io import read
from ase.visualize import view
from aiida import orm
from aiida_user_addons.process.transform import make_vac, make_supercell, rattle, get_primitive
from aiida_user_addons.tools.pymatgen import load_mp_struct

In [3]:
basepath = GroupPathX('mct-defect')
workpath = basepath['workflows']


## Compute Elemental references

In [12]:
structure = load_mp_struct('mp-19')
upd = VaspRelaxUpdater().apply_preset(structure, 
                                      code='vasp-6.3.2@sugon-xh-v2',
                                      overrides={'ispin': 1}
                                     )
upd.set_resources(tot_num_mpiprocs=16, num_machines=1)
upd.set_options(max_wallclock_seconds=3600 * 2, queue_name='xhhctdnormal')
upd.set_label('Te RELAX')

upd.builder
running = upd.submit()
workpath.add_node(running, 'te_elemental', True)

## Compute HgTe

In [63]:
hgte = read('HgTe.cif')
upd = VaspRelaxUpdater().apply_preset(orm.StructureData(ase=hgte), 
                                      code='vasp-6.3.2@sugon-xh-v2',
                                      overrides={'ispin': 1}
                                     )
upd.set_resources(tot_num_mpiprocs=32, num_machines=1)
upd.set_options(max_wallclock_seconds=3600, queue_name='xhhctdnormal')
upd.set_label('HgTe RELAX')
upd.set_kspacing(0.03)
upd.set_incar(sigma=0.2)

upd.builder

running = upd.submit()
# Note that this is actually the 8 atom conventional cell
workpath.add_node(running, 'hgte_primitive_k_spacing_0_03' + '_sigma_0_2', True)

Compute CdTe

In [10]:
cdte = read('CdTe.cif')
cdte_prim =  orm.StructureData(ase=cdte)
upd = VaspRelaxUpdater().apply_preset(cdte_prim, 
                                      code='vasp-6.3.2@sugon-xh-v2',
                                      overrides={'ispin': 1}
                                     )
upd.set_resources(tot_num_mpiprocs=16, num_machines=1)
upd.set_options(max_wallclock_seconds=3600, queue_name='xhhctdnormal')
upd.set_label('CdTe RELAX')
upd.builder

running = upd.submit()

workpath.add_node(running, 'cdte_primitive', True)

## Analyse results

In [64]:
workpath['hgte_primitive'].get_node().outputs.misc['total_energies']['energy_extrapolated']

-17.50050979

In [65]:
workpath['cd_elemental'].get_node().outputs.misc['total_energies']['energy_extrapolated']

-2.40974002

In [65]:
workpath['cd_elemental'].get_node().outputs.misc['total_energies']['energy_extrapolated']

-2.40974002

In [71]:
workpath['cd_elemental_kspacing_0_03'].get_node().outputs.misc['total_energies']['energy_extrapolated']

-2.3815042

In [72]:
workpath['cd_elemental_kspacing_0_03_sigma_0_2'].get_node().outputs.misc['total_energies']['energy_extrapolated']

-2.38167658

In [76]:
workpath['hg_elemental'].get_node().outputs.misc['total_energies']['energy_extrapolated']

-1.64902925

In [122]:
workpath['hg_elemental_k_spacing_0_03'].get_node().outputs.misc['total_energies']['energy_extrapolated']

-1.66184167

In [125]:
workpath['hg_elemental_k_spacing_0_03_sigma_0_2'].get_node().outputs.misc['total_energies']['energy_extrapolated']

-1.66315616

In [78]:
workpath['hgte_primitive'].get_node().outputs.misc['total_energies']['energy_extrapolated']

-17.50050979

In [124]:
workpath['hgte_primitive_k_spacing_0_03_sigma_0_2'].get_node().outputs.misc['total_energies']['energy_extrapolated']

-17.50976607

Comment:

1. 0.05 kspacing is sufficient for Cd and HgTe, but not for Hg
2. However, using 0.05 will not change the energy of elemental Hg significantly enough to affect the formation energy of $V_{Hg}$

In [216]:
workpath.show_tree(decorate_with_exit_status)

workflows
├── cd_elemental [0]
├── cd_elemental_kspacing_0_03 [0]
├── cd_elemental_kspacing_0_03_sigma_0_2 [0]
├── cd_elemental_soc_nosym [0]
├── hg_elemental [0]
├── hg_elemental_k_spacing_0_03 [0]
├── hg_elemental_k_spacing_0_03_sigma_0_2 [0]
├── hg_elemental_soc [0]
├── hg_elemental_soc_nosym [0]
├── hgte222_V_hg [0]
├── hgte222_V_hg_rattle [killed]
├── hgte222_V_hg_soc_nosym [0]
├── hgte222_supercell [0]
├── hgte222_supercell_soc_nosym [0]
├── hgte333_V_hg_gam [0]
├── hgte333_supercell [0]
├── hgte_primitive [0]
├── hgte_primitive_k_spacing_0_03 [0]
├── hgte_primitive_k_spacing_0_03_sigma_0_2 [0]
├── hgte_true_primitimve_333_supercell  [0]
├── hgte_true_primitive [0]
└── hgte_true_primitive_333_V_hg [0]



## Proceed with defect calculation

In [4]:
vac_cell = make_vac( workpath['hgte_primitive'].get_node().outputs.relax.structure,[0], [2,2,2])

upd = VaspRelaxUpdater().apply_preset(vac_cell, 
                                      code='vasp-6.3.2@sugon-xh-v2',
                                      overrides={'ispin': 2,
                                                 'nupdown': 2,
                                                 'isym': 0,
                                                 'ncore':8, 'kpar':4, 'lorbit': None}
                                     )
upd.set_resources(tot_num_mpiprocs=128, num_machines=2)
upd.set_options(max_wallclock_seconds=3600 * 12, queue_name='xhhctdnormal')
upd.set_label('HgTe PRIM 222 V_Hg RELAX')
upd.set_relax_settings(volume=False, shape=False)  # Relax only ionic positions
upd.builder

running = upd.submit()

workpath.add_node(running, 'hgte_primitive_2_V_hg_nupdown_2_isym0')

06/03/2025 02:30:33 PM <2951440> aiida.engine.processes.functions: [INFO] Executing process function, current stack status: 25 frames of 3000
06/03/2025 02:30:33 PM <2951440> aiida.orm.nodes.process.calculation.calcfunction.CalcFunctionNode: [INFO] Process<667737>: Broadcasting state change: state_changed.created.running
06/03/2025 02:30:33 PM <2951440> aiida.orm.nodes.process.calculation.calcfunction.CalcFunctionNode: [INFO] Process<667737>: Broadcasting state change: state_changed.running.finished


In [5]:
vac_cell = make_vac( workpath['hgte_primitive'].get_node().outputs.relax.structure,[0], [2,2,2])

upd = VaspRelaxUpdater().apply_preset(vac_cell, 
                                      code='vasp-6.3.2@sugon-xh-v2',
                                      overrides={'ispin': 2,
                                                 'isym': 0,
                                                 'ncore':8, 'kpar':4, 'lorbit': None}
                                     )
upd.set_resources(tot_num_mpiprocs=128, num_machines=2)
upd.set_options(max_wallclock_seconds=3600 * 12, queue_name='xhhctdnormal')
upd.set_label('HgTe PRIM 222 V_Hg RELAX')
upd.set_relax_settings(volume=False, shape=False)  # Relax only ionic positions
upd.builder

running = upd.submit()

workpath.add_node(running, 'hgte_primitive_2_V_hg_ispin_2_isym0')

06/03/2025 02:30:52 PM <2951440> aiida.engine.processes.functions: [INFO] Executing process function, current stack status: 25 frames of 3000
06/03/2025 02:30:52 PM <2951440> aiida.orm.nodes.process.calculation.calcfunction.CalcFunctionNode: [INFO] Process<667759>: Broadcasting state change: state_changed.created.running
06/03/2025 02:30:52 PM <2951440> aiida.orm.nodes.process.calculation.calcfunction.CalcFunctionNode: [INFO] Process<667759>: Broadcasting state change: state_changed.running.finished


In [134]:
ref_333 = make_supercell(workpath['hgte_primitive'].get_node().outputs.relax.structure,[3,3,3])['structure']

upd = VaspRelaxUpdater().apply_preset(ref_333, 
                                      code='vasp-6.3.2-gam@sugon-xh-v2',
                                      overrides={'ispin': 1,
                                                 'ncore':8, 'kpar':1, 'lorbit': None,}
                                     )
upd.set_resources(tot_num_mpiprocs=128, num_machines=2)
upd.set_options(max_wallclock_seconds=3600 * 12, queue_name='xhhctdnormal')
upd.set_label('HgTe 333 GAMMMA REF')
upd.set_relax_settings(volume=False, shape=False)  # Relax only ionic positions
upd.set_kpoints_mesh((1,1,1), (0, 0, 0))
upd.builder

running = upd.submit()

workpath.add_node(running, 'hgte333_supercell')

05/30/2025 11:18:10 PM <2182919> aiida.engine.processes.functions: [INFO] Executing process function, current stack status: 25 frames of 3000
05/30/2025 11:18:10 PM <2182919> aiida.orm.nodes.process.calculation.calcfunction.CalcFunctionNode: [INFO] Process<662081>: Broadcasting state change: state_changed.created.running
05/30/2025 11:18:10 PM <2182919> aiida.orm.nodes.process.calculation.calcfunction.CalcFunctionNode: [INFO] Process<662081>: Broadcasting state change: state_changed.running.finished


In [ ]:
ref_222 = make_supercell(workpath['hgte_primitive'].get_node().outputs.relax.structure,[2,2,2])['structure']

upd = VaspRelaxUpdater().apply_preset(ref_222, 
                                      code='vasp-6.3.2@sugon-xh-v2',
                                      overrides={'ispin': 2,
                                                 'ncore':8, 'kpar':2, 'lorbit': None,}
                                     )
upd.set_resources(tot_num_mpiprocs=128, num_machines=2)
upd.set_options(max_wallclock_seconds=3600 * 12, queue_name='xhhctdnormal')
upd.set_label('HgTe 222 SUPERCELL')
upd.set_relax_settings(volume=False, shape=False)  # Relax only ionic positions
upd.builder

running = upd.submit()

workpath.add_node(running, 'hgte222_supercell', True)

In [206]:
ref_222 = make_supercell(workpath['hgte_true_primitive'].get_node().outputs.relax.structure,[3,3,3])['structure']

upd = VaspRelaxUpdater().apply_preset(ref_222, 
                                      code='vasp-6.3.2@sugon-xh-v2',
                                      overrides={'ispin': 2,
                                                 'ncore':8, 'kpar':2, 'lorbit': None,}
                                     )
upd.set_resources(tot_num_mpiprocs=128, num_machines=2)
upd.set_options(max_wallclock_seconds=3600 * 12, queue_name='xhhctdnormal')
upd.set_label('HgTe PRIM 333 SUPERCELL')
upd.set_relax_settings(volume=False, shape=False)  # Relax only ionic positions
upd.builder

running = upd.submit()

workpath.add_node(running, 'hgte_true_prim_333_supercell', True)

05/31/2025 07:29:14 PM <2182919> aiida.engine.processes.functions: [INFO] Executing process function, current stack status: 25 frames of 3000
05/31/2025 07:29:14 PM <2182919> aiida.orm.nodes.process.calculation.calcfunction.CalcFunctionNode: [INFO] Process<662365>: Broadcasting state change: state_changed.created.running
05/31/2025 07:29:14 PM <2182919> aiida.orm.nodes.process.calculation.calcfunction.CalcFunctionNode: [INFO] Process<662365>: Broadcasting state change: state_changed.running.finished


## Add spin orbit coupling

Check the effect of SOC - do a single point energy calculation using the final relaxed structures

In [174]:
def gen_soc_sp(node, label):
    """Prepare a SOC singlepoint calculation"""
    upd = VaspBuilderUpdater().apply_preset(node, 
                                          code='vasp-6.3.2-ncl@sugon-xh-v2',
                                          overrides={'ispin': 1,
                                                     'lsorbit': True,
                                                     'isym': -1,
                                                     'ncore':8, 'kpar':2, 'lorbit': None,}
                                         )
    upd.set_resources(tot_num_mpiprocs=128, num_machines=2)
    upd.set_options(max_wallclock_seconds=3600 * 12, queue_name='xhhctdnormal')
    upd.set_label(label + ' SOC SP')
    upd.builder
    return upd

In [175]:
node = workpath['hg_elemental'].get_node().outputs.relax.structure
upd = gen_soc_sp(node, 'Hg RELAXED')
upd.set_resources(tot_num_mpiprocs=32, num_machines=1)
upd.builder
running = upd.submit()
workpath.add_node(running, 'hg_elemental_soc_nosym', True)

In [179]:
node = workpath['hgte222_supercell'].get_node().outputs.relax.structure
upd = gen_soc_sp(node, 'HgTe SUPERCELL')
upd.set_resources(tot_num_mpiprocs=128, num_machines=2)
upd.builder
running = upd.submit()
workpath.add_node(running, 'hgte222_supercell_soc_nosym', True)

In [180]:
node = workpath['hgte222_V_hg'].get_node().outputs.relax.structure
upd = gen_soc_sp(node, 'HgTe V_Hg SUPERCELL')
upd.set_resources(tot_num_mpiprocs=128, num_machines=2)
upd.builder
running = upd.submit()
workpath.add_node(running, 'hgte222_V_hg_soc_nosym', True)

## Compute the formation energy

In [106]:
def read_energy(path):
    return path.get_node().outputs.misc['total_energies']['energy_extrapolated']
def read_energy_per_atom(path):
    node = path.get_node()
    eng = node.outputs.misc['total_energies']['energy_extrapolated']
    return eng / len(node.inputs.structure.sites)

In [239]:
def show_formation_energy(supercell, v_hg, elemental):
    evac = read_energy(workpath[v_hg])
    print(f'Vacancy bearing cell: {evac:.5f} eV')
    ebulk = read_energy(workpath[supercell])
    print(f'Bulk cell: {ebulk: .5f} eV')
    e_hg = read_energy_per_atom(workpath[elemental])
    print(f'Energy per Hg atom: {e_hg: .5f} eV')
    e_vac = evac + e_hg - ebulk
    print(f'-->Vacancy formation energy: {e_vac:.5f} eV')

### Primitive cell 333 supercell - 54 atoms

In [240]:
show_formation_energy('hgte_true_primitive_333_supercell', 'hgte_true_primitive_333_V_hg', 'hg_elemental')

Vacancy bearing cell: -115.95585 eV
Bulk cell: -117.83946 eV
Energy per Hg atom: -0.54968 eV
-->Vacancy formation energy: 1.33394 eV


### Standard 222 cell

In [242]:
show_formation_energy('hgte222_supercell', 'hgte222_V_hg', 'hg_elemental')

Vacancy bearing cell: -138.18654 eV
Bulk cell: -140.00824 eV
Energy per Hg atom: -0.54968 eV
-->Vacancy formation energy: 1.27202 eV


### Standard 222 cell + SOC

In [243]:
show_formation_energy('hgte222_supercell_soc_nosym', 'hgte222_V_hg_soc_nosym', 'hg_elemental_soc_nosym')

Vacancy bearing cell: -150.87398 eV
Bulk cell: -152.88934 eV
Energy per Hg atom: -0.79927 eV
-->Vacancy formation energy: 1.21609 eV


### Standard 333 cell (Gamma-only calculations)

In [244]:
show_formation_energy('hgte333_supercell', 'hgte333_V_hg_gam', 'hg_elemental')

Vacancy bearing cell: -469.91267 eV
Bulk cell: -471.45545 eV
Energy per Hg atom: -0.54968 eV
-->Vacancy formation energy: 0.99310 eV


Comments:

1. The Hg_vacancy formation energy is about 1 eV with large supercell size at the Hg-rich limit.
2. SOC does not appear to significantly alter the formation energy (less than 0.1 eV)
3. Size of the supercell can significantly alter the formation energy, but we are limited to small size as many materials are involved

## Check energy and chemical formula using show_tree

This can be used to manually validate the vacancy formation energy calculated

In [247]:
def form(path):
    if path.is_node:
        return path.get_node().inputs.structure.get_formula()
def energy(path):
    if path.is_node:
        if not path.get_node().is_finished_ok:
            return
        return '{:.4f} eV'.format(path.get_node().outputs.misc['total_energies']['energy_extrapolated'])

In [248]:
workpath.show_tree(form, energy)

workflows
├── cd_elemental Cd2 | -2.4097 eV
├── cd_elemental_kspacing_0_03 Cd2 | -2.3815 eV
├── cd_elemental_kspacing_0_03_sigma_0_2 Cd2 | -2.3817 eV
├── cd_elemental_soc_nosym Cd2 | -2.4699 eV
├── hg_elemental Hg3 | -1.6490 eV
├── hg_elemental_k_spacing_0_03 Hg3 | -1.6618 eV
├── hg_elemental_k_spacing_0_03_sigma_0_2 Hg3 | -1.6632 eV
├── hg_elemental_soc Hg3 | -2.3978 eV
├── hg_elemental_soc_nosym Hg3 | -2.3978 eV
├── hgte222_V_hg Hg31Te32 | -138.1865 eV
├── hgte222_V_hg_rattle Hg31Te32
├── hgte222_V_hg_soc_nosym Hg31Te32 | -150.8740 eV
├── hgte222_supercell Hg32Te32 | -140.0082 eV
├── hgte222_supercell_soc_nosym Hg32Te32 | -152.8893 eV
├── hgte333_V_hg_gam Hg107Te108 | -469.9127 eV
├── hgte333_supercell Hg108Te108 | -471.4554 eV
├── hgte_primitive Hg4Te4 | -17.5005 eV
├── hgte_primitive_k_spacing_0_03 Hg4Te4 | -17.5093 eV
├── hgte_primitive_k_spacing_0_03_sigma_0_2 Hg4Te4 | -17.5098 eV
├── hgte_true_primitive HgTe | -4.3643 eV
├── hgte_true_primitive_333_V_hg Hg26Te27 | -115.9559 eV
└

In [8]:
workpath.browse.hgte222_supercell().node

<WorkChainNode: uuid: 4495b396-efa3-46bf-9b0d-4fa73bffafb1 (pk: 662173) (aiida.workflows:vasp.relax)>